# Interacting With Geospatial Tools and Data on DASH

## Learning Objectives
- Understand how to access the Geospatial tools available on the DASH platform
- Connect to the datalake from these tools and interact with the data
- Carry out a Geospatial Workflow

## ArcGIS Pro

ArcGIS Pro is a full-featured professional desktop GIS application from Esri. With ArcGIS Pro, you can explore, visualize, and analyze data; create 2D maps and 3D scenes; and share your work to ArcGIS Online or your ArcGIS Enterprise portal.

### How to access ArcGIS Pro
- Bring your own licence
- Accessed through the [Azure Virtual Desktop (AVD)](https://windows.cloud.microsoft/#/devices)
  * You can be granted access to a shared AVD and a Geospatial AVD

### How to Access Data in ArcGIS Pro
- Connect to Cloud Store (raster only)
- Use Azure Storage Explorer
- ArcGIS Online

Once a project has been set up in ArcGIS Pro, connect to the cloud store following the instructions in the Playbook [here](https://didactic-doodle-58b6846e.pages.github.io/docs/arcgis.html). For the exercise detailed in this training, import the UKCEH 2023 10m resolution Land Classification layer from Unrestricted/source_uk_ceh_environmental_info_data_centre

Then, access azure storage explorer using the instructions [here](https://didactic-doodle-58b6846e.pages.github.io/docs/azure_storage_explorer.html) to pull in the SSSI layer in json format from the base/unrestricted/source_defra_data_services_platform/ directory. Download to your ArcGIS Project folder. 

Follow the instructions in the accompanying training recording to process the raster layer and calculate the proportion of each habitat within each SSSI. 

## Databricks

- Pull in data
- Run similar Geospatial workflow for national dataset
- Publish to ArcGIS Online
- View in ArcGIS Pro / AGOL

### Python Workflow

In [0]:
%pip install geopandas rasterio pandas numpy
%pip install shapely  
%pip install arcgis
%pip install shapely fiona
%pip install pyshp==2.1.3
%pip install keplergl
%pip install rasterstats

In [0]:
%restart_python

In [0]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import pandas as pd
import numpy as np
from rasterstats import zonal_stats

In [0]:
# Read in our raster dataset
src = rasterio.open('/Volumes/prd_dash_bronze/uk_ceh_environmental_info_data_centre_unrestricted/land_cover_map_2023_10m/format_GEOTIFF_land_cover_map_2023_10m/LATEST_land_cover_map_2023_10m/gblcm2023_10m.tif')

# Read in the SSSI layer
sssi_areas = gpd.read_parquet('/Volumes/prd_dash_bronze/defra_data_services_platform_unrestricted/sites_of_special_scientific_interest/format_GEOPARQUET_sites_of_special_scientific_interest/LATEST_sites_of_special_scientific_interest/Sites_of_Special_Scientific_Interest_England.parquet')

# create subset of data that matches that of the ArcGIS workflow
sssi_select = sssi_areas.loc[sssi_areas['name'].isin(['Bratton Downs SSSI', 'Salisbury Plain SSSI'])]

# Set crs
sssi_areas = sssi_areas.to_crs(epsg=27700)
sssi_areas

In [0]:
# Check crs matches between layers
print(sssi_areas.crs)
print(src.crs)

In [0]:
def calculate_landcover_stats(gdf, raster_src, name_field="name"):
    """
    Calculate land cover statistics (pixel counts and area) for each polygon
    in a GeoDataFrame using a raster dataset.

    Parameters
    ----------
    gdf : GeoDataFrame
        Input polygons (e.g. SSSI boundaries)
    raster_src : rasterio DatasetReader
        Open raster dataset (e.g. land cover raster)
    name_field : str, optional
        Column name containing polygon identifier (default = "name")

    Returns
    -------
    results : list of dict
        Each dict contains:
            - polygon name
            - land cover class
            - pixel count
            - area in m²
    """

    results = []

    # Calculate pixel area
    pixel_area = raster_src.res[0] * raster_src.res[1]
    nodata = raster_src.nodata

    # Loop through each polygon
    for idx, row in gdf.iterrows():
        geom = [row.geometry]

        try:
            # Clip raster to polygon
            out_image, _ = mask(raster_src, geom, crop=True)

            # Extract first band
            data = out_image[0]

            # Remove nodata values
            if nodata is not None:
                data = data[data != nodata]

            # Skip empty results
            if data.size == 0:
                continue

            # Get unique classes and counts
            classes, counts = np.unique(data, return_counts=True)

            # Convert counts to area
            areas = counts * pixel_area

            # Store results
            for c, count, area in zip(classes, counts, areas):
                results.append({
                    "sssi_name": row[name_field],
                    "land_class": int(c),
                    "pixel_count": int(count),
                    "area_m2": float(area)
                })

        except Exception as e:
            print(f"Skipping feature {idx}: {e}")

    return results

In [0]:
# call function and assign output to variable
results = calculate_landcover_stats(sssi_select, src, name_field="name")

# Remove 0 class values i.e. those outside the SSSI but unclassified
df = pd.DataFrame(results)
df = df[df['land_class'] != 0]

display(df)

In [0]:
# Create pivot table so that SSSI names are columns
pivot = df.pivot_table(
    index='land_class',
    columns='sssi_name',
    values='area_m2',
    fill_value=0
)

display(pivot)

In [0]:
# Call function again and run for whole SSSI dataset
results_national = calculate_landcover_stats(sssi_areas, src)

# Convert to dataframe
df_nat = pd.DataFrame(results_national)
df_nat = df_nat[df_nat['land_class'] != 0]

# create pivot
pivot = df_nat.pivot_table(
    index='sssi_name',
    columns='land_class',
    values='area_m2',
    fill_value=0
)

display(pivot)

In [0]:
# Read in csv file containing land class code and names
csv = pd.read_csv('/Volumes/prd_dash_lab/dash_training_unrestricted/training/ArcGIS/LCM_Class_Table.csv')

# Convert to dictionary
lc_dict = dict(zip(csv['LC_Identifier'], csv['LC_Class']))
lc_dict

In [0]:
# Rename table headers by passing the dictionary into the function
pivot_headers = pivot.rename(columns=lc_dict)
display(pivot_headers.reset_index())

In [0]:
# Convert sqm to proportion of land cover
lcm_prop = pivot_headers.div(pivot_headers.sum(axis=1), axis=0)
lcm_perc = lcm_prop * 100
display(lcm_perc.reset_index())

In [0]:
# Re-join output table back to the original SSSI layer on the SSSI name field
sssi_lcm_join = sssi_areas.merge(
    lcm_perc,
    left_on="name",
    right_on="sssi_name",
    how="left"
)

display(sssi_lcm_join)

In [0]:
# tidy up column names to not contain spaces
sssi_lcm_join.columns = [
    col.replace(' ', '_') for col in sssi_lcm_join.columns
]

# Ensure type is geodataframe with correct geometry column
sssi_lcm_join = sssi_lcm_join.set_geometry("geometry")
sssi_join_gdf = gpd.GeoDataFrame(sssi_lcm_join, geometry='geometry')

# Remove na values
sssi_join_gdf.fillna(0)

# Drop unnecessary columns
sssi_join_gdf = sssi_join_gdf.drop(columns=['hyperlink', 'contact_no'])
display(sssi_join_gdf)

In [0]:
from keplergl import KeplerGl

def kepler_landcover_map(
    gdf,
    value_column,
    name_column="name",
    simplify_tol=0.0001,
    map_height=600
):

    if value_column not in gdf.columns:
        raise ValueError(f"{value_column} not found in GeoDataFrame")

    df = gdf[[name_column, value_column, "geometry"]].copy()
    df = df.to_crs(epsg=4326)


    df["value"] = pd.to_numeric(df[value_column], errors="coerce")


    df["geometry"] = df["geometry"].simplify(simplify_tol, preserve_topology=True)

    geojson_data = df.__geo_interface__

    config = {
        "version": "v1",
        "config": {
            "visState": {
                "layers": [
                    {
                        "id": "sssi_layer",
                        "type": "geojson",
                        "config": {
                            "dataId": "SSSI",
                            "label": value_column,
                            "columns": {
                                "geojson": "geometry"
                            },
                            "isVisible": True,
                            "visConfig": {
                                "opacity": 0.7,
                                "filled": True,
                                "stroked": True,
                                "thickness": 0.5
                            }
                        },
                        "visualChannels": {
                            "colorField": {
                                "name": "value",
                                "type": "real"
                            },
                            "colorScale": "quantile"
                        }
                    }
                ]
            }
        }
    }

    map_ = KeplerGl(height=map_height, config=config)

    map_.add_data(geojson_data, name="SSSI")

    return map_

In [0]:
kepler_landcover_map(sssi_join_gdf.head(2000), "Deciduous_woodland")

### Upload to AGOL

In [0]:
# Load in the ArcGIS module and log in using credentials
from arcgis.gis import GIS

username = dbutils.secrets.get(scope='arcgis_scope', key='username')
password = dbutils.secrets.get(scope='arcgis_scope', key='password')

gis = GIS(
    url='https://defra.maps.arcgis.com/',
    username=username,
    password=password
)

print("Connected to:", gis.properties.name)

In [0]:
# simplify geometry (optional)
sssi_join_gdf["geometry"] = sssi_join_gdf["geometry"].simplify(0.001, preserve_topology=True)

In [0]:
import os
import shutil

# Create a folder on the local disk
folder = "/local_disk0/sssi_simple_folder"
os.makedirs(folder, exist_ok=True)

# Save shapefile and associated side car files into folder
sssi_join_gdf.to_file(f"{folder}/sssi_simple.shp")

# zip the folder
shutil.make_archive("/local_disk0/sssi_simple", "zip", folder)


In [0]:
# Upload the shapefile to ArcGIS Online
item = gis.content.add(
    {"title": "SSSI Join Simple Upload", "type": "Shapefile"},
    data="/local_disk0/sssi_simple.zip"
)

# Create feature layer by publishing
published = item.publish()

print(published.url)

In [0]:
%skip
from keplergl import KeplerGl

gdf = sssi_join_gdf[[
    "name",
    "Deciduous_woodland",
    "geometry"
]].copy()

gdf = gdf.to_crs(epsg=4326)

gdf["geometry"] = gdf["geometry"].simplify(0.001)

gdf = gdf[["name", "Deciduous_woodland", "geometry"]]

geojson_data = gdf.__geo_interface__

map_ = KeplerGl(height=600)
map_.add_data(geojson_data, "SSSI")

map_